# Chapter 1 — Introduction: Earth Observation + Deep Learning

## Learning Objectives
- Understand what satellite imagery is and why it matters for AI
- Load and visualise multi-band GeoTIFF files with rasterio
- Understand spectral bands, spatial resolution, and CRS
- Explore the EuroSAT dataset structure
- Build intuition for how CNNs can extract land-use information

## Estimated Duration
Theory: 2h | Practical: 2h | Total: 4h

## Difficulty: Beginner

## Datasets Used
- EuroSAT RGB (auto-download via torchgeo, ~90MB)

## Key Concepts
- Satellites and EO data: Sentinel-2, Landsat, spatial resolution
- Spectral bands: visible, NIR, SWIR, how they relate to land features
- Raster vs vector data in GIS
- Why CNNs work for image classification
- What makes EO different from natural image datasets

In [ ]:
# ─── COLAB SETUP (run this cell first) ────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q torch torchvision torchgeo rasterio rioxarray geopandas matplotlib seaborn

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = './data' if not IN_COLAB else '/content/data'
print(f'Device: {DEVICE} | Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}')

## 1.1 — What Is Earth Observation?

Earth Observation (EO) is the systematic collection of information about the
Earth's physical, chemical, and biological systems from remote sensors —
primarily satellites but also aircraft and drones.

Key EO satellites:
| Satellite    | Operator | Resolution | Revisit | Bands |
|-------------|----------|------------|---------|-------|
| Sentinel-2  | ESA      | 10-60m     | 5 days  | 13    |
| Landsat 8/9 | USGS/NASA| 30m        | 16 days | 11    |
| WorldView-3 | Maxar    | 0.3m       | 1 day   | 29    |
| Planet NICFI| Planet   | 4.77m      | daily   | 4     |

Why does spatial resolution matter for CNNs?
- At 10m/pixel, a single Sentinel-2 pixel covers a tennis court
- At 0.3m/pixel, individual cars and tree crowns are visible
- CNN kernel design must account for the scale of objects of interest

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Visualise what 10m vs 30m resolution looks like
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Simulate resolution pyramid
resolutions = [1, 3, 10]  # pixel size ratios
titles = ['High res (0.5m, Pleiades)', 'Medium res (10m, Sentinel-2)', 'Low res (30m, Landsat)']

np.random.seed(42)
base = np.random.rand(100, 100, 3)
# Add some structure to simulate land features
base[20:40, 20:40] = [0.2, 0.5, 0.2]   # forest patch
base[60:80, 60:80] = [0.7, 0.6, 0.5]   # urban patch
base[10:20, 70:90] = [0.2, 0.3, 0.8]   # water body

from PIL import Image
for ax, r, title in zip(axes, resolutions, titles):
    # Simulate downsampling then upsampling (pixelation effect)
    small = base[::r, ::r]
    display = np.repeat(np.repeat(small, r, axis=0), r, axis=1)[:100, :100]
    ax.imshow(np.clip(display, 0, 1))
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle('Spatial Resolution: Same Area at Different GSD (Ground Sampling Distance)', fontsize=12)
plt.tight_layout()
plt.show()

## 1.2 — Spectral Bands: What Satellites Actually Measure

Satellites don't take 'photos' — they measure electromagnetic radiation
reflected from the Earth's surface at specific wavelength ranges.

Sentinel-2 Band Reference:
| Band  | Name         | λ (nm)    | Resolution | Use case                    |
|-------|-------------|-----------|------------|-----------------------------|
| B01   | Coastal     | 442.7     | 60m        | Atmospheric correction      |
| B02   | Blue        | 492.7     | 10m        | RGB visualisation           |
| B03   | Green       | 559.8     | 10m        | RGB visualisation           |
| B04   | Red         | 664.6     | 10m        | RGB, vegetation stress      |
| B05   | Red Edge 1  | 704.1     | 20m        | Vegetation chlorophyll      |
| B06   | Red Edge 2  | 740.5     | 20m        | Canopy chlorophyll content  |
| B07   | Red Edge 3  | 782.8     | 20m        | Leaf area index             |
| B08   | NIR         | 832.8     | 10m        | Vegetation (NDVI)           |
| B8A   | NIR Narrow  | 864.7     | 20m        | Water vapour reference      |
| B09   | Water Vapour| 945.1     | 60m        | Water vapour correction     |
| B10   | SWIR Cirrus | 1373.5    | 60m        | Cirrus cloud detection      |
| B11   | SWIR 1      | 1613.7    | 20m        | Snow/ice, soil moisture     |
| B12   | SWIR 2      | 2202.4    | 20m        | Geology, soil mapping       |

The key insight for CNNs: EACH BAND is an IMAGE. A 13-band Sentinel-2 scene
is a (13, H, W) tensor — identical in structure to how a CNN sees input images.

In [ ]:
# Simulate how different bands reveal different land features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

np.random.seed(42)
H, W = 128, 128

# Simulate a scene with urban, forest, agriculture, water patches
scene = np.zeros((H, W), dtype=np.float32)
scene[20:60, 20:60] = 1.0   # Urban
scene[70:110, 20:60] = 2.0  # Forest
scene[20:60, 70:110] = 3.0  # Agriculture
scene[70:110, 70:110] = 4.0 # Water

# Spectral response per land cover class [Blue, Green, Red, NIR, SWIR1, SWIR2, RedEdge]
spectral = {
    'Urban':       [0.08, 0.09, 0.10, 0.12, 0.15, 0.12, 0.11],
    'Forest':      [0.02, 0.06, 0.04, 0.45, 0.25, 0.10, 0.35],
    'Agriculture': [0.03, 0.08, 0.06, 0.35, 0.28, 0.15, 0.28],
    'Water':       [0.06, 0.07, 0.04, 0.01, 0.01, 0.005, 0.02],
}

band_names = ['Blue (B02)', 'Green (B03)', 'Red (B04)', 'NIR (B08)', 
              'RedEdge (B05)', 'SWIR1 (B11)', 'SWIR2 (B12)', 'NDVI']

class_map = {0: 'background', 1: 'Urban', 2: 'Forest', 3: 'Agriculture', 4: 'Water'}
class_vals = [0.05, *[spectral[k] for k in ['Urban', 'Forest', 'Agriculture', 'Water']]]

bands_data = []
for band_idx in range(7):
    band = np.ones((H, W), dtype=np.float32) * 0.05
    for cls_val, (cls_name, spectrum) in zip([1, 2, 3, 4], spectral.items()):
        band[scene == cls_val] = spectrum[band_idx]
    band += np.random.normal(0, 0.005, (H, W))  # sensor noise
    bands_data.append(band)

# NDVI = (NIR - Red) / (NIR + Red)
nir, red = bands_data[3], bands_data[2]
ndvi = (nir - red) / (nir + red + 1e-8)
bands_data.append(ndvi)

cmaps = ['Blues', 'Greens', 'Reds', 'YlOrRd', 'Purples', 'hot', 'copper', 'RdYlGn']
for ax, band, name, cmap in zip(axes.ravel(), bands_data, band_names, cmaps):
    im = ax.imshow(band, cmap=cmap)
    ax.set_title(name, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    'Same Scene Across Different Spectral Bands\n'
    '(Urban=top-left, Forest=bottom-left, Agriculture=top-right, Water=bottom-right)',
    fontsize=12
)
plt.tight_layout()
plt.show()

print("Key observation:")
print("- Vegetation has HIGH NIR reflectance (B08) — this is why NDVI works")
print("- Water absorbs almost all NIR/SWIR light — appears nearly black in those bands")
print("- Urban areas have relatively uniform spectral response — similar reflectance across bands")
print("- NDVI highlights vegetation (red/green) and separates water (dark) from urban (neutral)")

## 1.3 — Loading the EuroSAT Dataset

EuroSAT (Helber et al., 2019) is the 'MNIST of EO' — a clean, balanced
benchmark dataset that makes CNN experimentation fast and reproducible.

27,000 Sentinel-2 image patches, 64×64 pixels, 10 land use classes:
  0: AnnualCrop    1: Forest        2: HerbaceousVegetation  3: Highway
  4: Industrial    5: Pasture       6: PermanentCrop          7: Residential
  8: River         9: SeaLake

We load it via torchgeo — a PyTorch library that handles EO dataset
download, CRS management, and dataset splitting.

In [ ]:
import os
from pathlib import Path
from torchgeo.datasets import EuroSAT
from torch.utils.data import DataLoader

# torchgeo will download EuroSAT automatically (~90MB) on first run
EUROSAT_ROOT = Path(DATA_ROOT) / 'eurosat'
EUROSAT_ROOT.mkdir(parents=True, exist_ok=True)

print('Loading EuroSAT dataset (downloading if needed)...')
dataset = EuroSAT(root=EUROSAT_ROOT, split='train', download=True)

print(f'\nDataset size: {len(dataset)} samples')
print(f'Classes ({len(dataset.classes)}): {dataset.classes}')

# Inspect a single sample
sample = dataset[0]
print(f'\nSample keys: {list(sample.keys())}')
print(f'Image shape: {sample["image"].shape}  (C×H×W)')
print(f'Image dtype: {sample["image"].dtype}')
print(f'Label: {sample["label"]} → {dataset.classes[sample["label"]]}')
print(f'\nPixel value range: [{sample["image"].min():.0f}, {sample["image"].max():.0f}]')
print('(Sentinel-2 L2A reflectance scaled to 0–10000 range)')

In [ ]:
# Visualise one sample from each class
from torchgeo.datasets import EuroSAT
import torch

dataset_full = EuroSAT(root=EUROSAT_ROOT, split='train', download=True)

# Collect one example per class
class_examples = {}
for sample in dataset_full:
    label = sample['label'].item()
    if label not in class_examples:
        class_examples[label] = sample['image']
    if len(class_examples) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for cls_idx, (label, img) in enumerate(sorted(class_examples.items())):
    ax = axes[cls_idx // 5, cls_idx % 5]
    # Display RGB (bands 3=Red, 2=Green, 1=Blue in Sentinel-2 order)
    # EuroSAT uses torchgeo band ordering: B02=idx0, B03=idx1, B04=idx2 for RGB version
    rgb = img[:3].permute(1, 2, 0).float()  # take first 3 channels
    # Percentile stretch for display
    for c in range(3):
        p2, p98 = torch.quantile(rgb[..., c], torch.tensor([0.02, 0.98]))
        rgb[..., c] = (rgb[..., c] - p2) / (p98 - p2 + 1e-8)
    rgb = rgb.clamp(0, 1)
    ax.imshow(rgb.numpy())
    ax.set_title(dataset_full.classes[label], fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('EuroSAT — One Sample per Class (Sentinel-2 RGB)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Explore class distribution
train_ds = EuroSAT(root=EUROSAT_ROOT, split='train', download=True)
val_ds = EuroSAT(root=EUROSAT_ROOT, split='val', download=True)
test_ds = EuroSAT(root=EUROSAT_ROOT, split='test', download=True)

print(f'Split sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

# Count labels
from collections import Counter
label_counts = Counter()
for sample in train_ds:
    label_counts[sample['label'].item()] += 1

print('\nClass distribution in training set:')
class_names = train_ds.classes
for idx, name in enumerate(class_names):
    count = label_counts[idx]
    bar = '█' * (count // 100)
    print(f'  {idx:2d} {name:<25} {count:5d} {bar}')

# Plot distribution
fig, ax = plt.subplots(figsize=(12, 4))
counts = [label_counts[i] for i in range(len(class_names))]
bars = ax.bar(class_names, counts, color='#2196F3', edgecolor='white')
ax.set_title('EuroSAT Training Set — Class Distribution')
ax.set_ylabel('Number of samples')
plt.xticks(rotation=45, ha='right')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(count), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nMin class count: {min(counts)}  |  Max: {max(counts)}')
print(f'Imbalance ratio: {max(counts)/min(counts):.2f}x')
print('EuroSAT is relatively balanced — good for beginners!')

## 1.4 — Why CNNs? Spatial Patterns in Satellite Imagery

Traditional ML (SVM, Random Forest) operates on per-pixel spectral signatures.
CNNs fundamentally differ: they learn SPATIAL PATTERNS across neighbourhoods.

For land cover classification:
- Forests: textured canopy patterns, uniform dark NIR response
- Urban areas: grid-like street patterns, high texture heterogeneity
- Water: smooth, homogeneous, specular reflection
- Agriculture: regular field boundaries, row-crop striping

None of these patterns are captured by per-pixel methods.
CNNs learn them automatically through hierarchical feature extraction.

In [ ]:
# Visualise texture differences between classes
import torch
from torch.utils.data import DataLoader

# Load a larger batch to find specific classes
loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
batch = next(iter(loader))
images, labels = batch['image'], batch['label']

target_classes = {3: 'Highway', 0: 'AnnualCrop', 9: 'SeaLake', 7: 'Residential'}
found = {}
for i, lbl in enumerate(labels):
    l = lbl.item()
    if l in target_classes and l not in found:
        found[l] = images[i]
    if len(found) == len(target_classes):
        break

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (cls_id, img) in zip(axes, found.items()):
    rgb = img[:3].permute(1, 2, 0).float()
    for c in range(3):
        p2, p98 = torch.quantile(rgb[..., c], torch.tensor([0.02, 0.98]))
        rgb[..., c] = (rgb[..., c] - p2) / (p98 - p2 + 1e-8)
    ax.imshow(rgb.clamp(0, 1).numpy())
    ax.set_title(target_classes[cls_id], fontsize=11, fontweight='bold')
    ax.axis('off')
    # Add a label about spatial pattern
    patterns = {
        3: 'Linear structure\n(road edges)',
        0: 'Regular rows\n(crop patterns)',
        9: 'Smooth, homogeneous\n(water surface)',
        7: 'Dense, heterogeneous\n(building mix)'
    }
    ax.set_xlabel(patterns.get(cls_id, ''), fontsize=9)

plt.suptitle('Spatial Patterns Visible in EuroSAT — What CNNs Learn', fontsize=12)
plt.tight_layout()
plt.show()

print("These textures and spatial arrangements are what CNN convolutional kernels will learn to detect.")

## Practical Exercises

### Exercise 1.1 — Dataset Exploration
Load the EuroSAT test split and compute:
- Mean pixel value per channel across all test samples
- Standard deviation per channel
- Print these as a table (these will become your normalization stats)

### Exercise 1.2 — Spectral Analysis
For 5 random samples from each class, compute and plot the mean
spectral profile (reflectance vs band number). You should see:
- Water: near-zero NIR/SWIR values
- Forest: high NIR reflectance
- Urban: relatively flat curve

### Exercise 1.3 — NDVI Computation
If using EuroSAT MS (13-band):
- Compute NDVI for 3 samples from 'Forest' and 3 from 'SeaLake'
- Display them as heatmaps
- What NDVI threshold separates the two classes?

### Mini-Project 1
Write a function `describe_sample(idx: int)` that prints:
- The class name
- The RGB and NIR mean values
- A side-by-side display of the RGB image and an NDVI heatmap
Run it on 5 samples from classes of your choice.

In [ ]:
# ─── EXERCISE STARTER CODE ────────────────────────────────────────────────────

# Exercise 1.1 — Compute channel statistics
def compute_dataset_stats(dataset, n_samples=1000):
    """
    Compute per-channel mean and standard deviation across a dataset.
    Use running statistics (no need to load everything into RAM).
    """
    # TODO: iterate over dataset, accumulate channel-wise sum and sum-of-squares
    # Hint: running_mean += image.mean(dim=[1,2]) / n_samples
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)
    
    channel_sum = None
    channel_sq_sum = None
    total_pixels = 0
    
    for batch in loader:
        images = batch['image'].float()  # (B, C, H, W)
        B, C, H, W = images.shape
        
        if channel_sum is None:
            channel_sum = torch.zeros(C)
            channel_sq_sum = torch.zeros(C)
        
        # Sum over batch, height, width
        channel_sum += images.sum(dim=[0, 2, 3])
        channel_sq_sum += (images ** 2).sum(dim=[0, 2, 3])
        total_pixels += B * H * W
    
    mean = channel_sum / total_pixels
    std = torch.sqrt(channel_sq_sum / total_pixels - mean ** 2)
    return mean.numpy(), std.numpy()

print('Computing training set statistics...')
mean, std = compute_dataset_stats(train_ds)

print('\nPer-channel statistics (use these for normalization):')
print(f'  Mean: {[f"{m:.1f}" for m in mean]}')
print(f'  Std:  {[f"{s:.1f}" for s in std]}')
print('\nNormalized (÷ 10000 to get 0-1 range):')
print(f'  Mean: {[f"{m/10000:.4f}" for m in mean]}')
print(f'  Std:  {[f"{s/10000:.4f}" for s in std]}')

In [ ]:
# Exercise 1.2 — Spectral profiles per class
import random

n_bands = train_ds[0]['image'].shape[0]
n_classes = len(train_ds.classes)
print(f'Number of bands in this EuroSAT variant: {n_bands}')

# NOTE: torchgeo's EuroSAT iterates samples in class-folder order
# (all AnnualCrop, then all Forest, ...). We must therefore either
# (a) shuffle indices before sampling, or (b) check the break
# condition against the full class set. We do BOTH for robustness
# and speed (avoids walking through the full 16k-sample dataset).
class_spectra = {cls_name: [] for cls_name in train_ds.classes}
samples_per_class = 20
class_counts = Counter({name: 0 for name in train_ds.classes})

rng = random.Random(42)
indices = list(range(len(train_ds)))
rng.shuffle(indices)

for idx in indices:
    sample = train_ds[idx]
    label = sample['label'].item()
    cls_name = train_ds.classes[label]
    if class_counts[cls_name] < samples_per_class:
        # Mean across spatial dimensions
        profile = sample['image'].float().mean(dim=[1, 2]).numpy()
        class_spectra[cls_name].append(profile)
        class_counts[cls_name] += 1
    # Break only once every class has reached the target count
    if all(v >= samples_per_class for v in class_counts.values()):
        break

assert len(class_counts) == n_classes, (
    f'Expected {n_classes} classes, got {len(class_counts)}'
)
assert all(v == samples_per_class for v in class_counts.values()), (
    f'Some classes are under-sampled: {dict(class_counts)}'
)

# Plot spectral profiles
fig, ax = plt.subplots(figsize=(12, 6))
colors_cls = plt.cm.tab10(np.linspace(0, 1, len(train_ds.classes)))

band_labels = [f'B{i+1}' for i in range(n_bands)]

for cls_name, color in zip(train_ds.classes, colors_cls):
    profiles = np.array(class_spectra[cls_name])
    mean_profile = profiles.mean(axis=0)
    std_profile = profiles.std(axis=0)
    x = range(n_bands)
    ax.plot(x, mean_profile, label=cls_name, color=color, linewidth=2)
    ax.fill_between(x, mean_profile - std_profile, mean_profile + std_profile,
                    color=color, alpha=0.1)

ax.set_xticks(range(n_bands))
ax.set_xticklabels(band_labels)
ax.set_xlabel('Band')
ax.set_ylabel('Mean Reflectance (DN)')
ax.set_title('Spectral Profiles per Land Use Class (EuroSAT)', fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Observation: SeaLake has very low reflectance in NIR/SWIR, while Forest peaks in NIR.')
print('These spectral signatures partially explain WHY CNNs can distinguish these classes.')

In [ ]:
# Chapter 1 summary
print('═' * 60)
print('CHAPTER 1 SUMMARY')
print('═' * 60)
print()
print('Key takeaways:')
print('  1. Satellite imagery = multi-band tensors, not just RGB photos')
print('  2. Each band measures a different wavelength — contains different info')
print('  3. Spatial resolution determines the scale of detectable objects')
print('  4. EuroSAT: 27k samples, 10 classes, ~balanced — great for CNN experiments')
print('  5. Land classes have distinct spectral AND spatial signatures')
print()
print('In the next chapter: HOW do CNNs extract these spatial patterns?')
print('We build convolution from scratch using NumPy and PyTorch.')